# u05_full_ft3 — train

Physical crop / coarse patch attention ablation. Gold58 development CV is not the public leaderboard.


In [ ]:
EXPERIMENT = {'id': 'u05_full_ft3', 'suite': 'weak', 'backbone': 'swin_t', 'feature_dim': 768, 'slices': 5, 'crop': 224, 'resize': 232, 'interpolation': 'bicubic', 'aggregation': 'class_mil', 'triplet': True, 'protocols': True, 'weak_labels': True, 'finetune': False, 'ensemble': False, 'backbone_epochs': 3, 'epochs': 20, 'batch_size': 16, 'lr': 0.001, 'weight_decay': 0.01, 'expected_training_studies': 4350, 'labels_sha256': '6f704a7bdb2f894cc49445b19ba7c4378c3f548d3449e00361e10044bee40920', 'train_dependencies': ['u03_patch_attention', 'a03_s04'], 'pipeline_sha256': '8ac712af29daeb144ab33c154660c00a73f7a3053ebb5f9e44fc071ea7d2d451', 'architecture': 'anatomy_v1', 'representation': 'single', 'neighbors': 5, 'centers': 5, 'hidden_dim': 128, 'heads': 4, 'sequence': True, 'position': 'rope_mm', 'plane_embedding': True, 'fusion': 'cross', 'mm_unit': 3.0, 'patch_attention': True, 'field_mm': 140.0, 'canonical': True, 'method': 'model', 'comparison_parent': 'u04_full_ft1', 'adapt_stage': 'a03_s04'}


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','pydicom==3.0.2','python-gdcm==3.2.6'])


In [ ]:
"""Shared source embedded into each standalone experiment notebook."""
from pathlib import Path
import hashlib
import json
import math
import numpy as np
import pandas as pd
import torch
from torch import nn
PLANES = ('Sagittal', 'Coronal', 'Axial')
LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
SEEDS = (42, 2026, 3407)

def file_hash(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

def validate_bundle(path, expected_config):
    path = Path(path).resolve()
    manifest = json.loads((path / 'assets.json').read_text(encoding='utf-8'))
    if manifest['experiment'] != expected_config:
        raise ValueError("Wrong experiment bundle: attach this experiment's completed training version.")
    files = manifest['files']
    wheels = {p.relative_to(path).as_posix() for p in (path / 'wheels').glob('*.whl')}
    declared_wheels = {name for name in files if name.startswith('wheels/') and name.endswith('.whl')}
    if not {'encoder.ts', 'model.pt'}.issubset(files) or wheels != declared_wheels or len(wheels) != 2 or (not any((name.startswith('wheels/pydicom-3.0.2-') for name in wheels))) or (not any((name.startswith('wheels/python_gdcm-3.2.6-') for name in wheels))):
        raise ValueError('Bundle must include hashes for both models and exactly the two required wheels.')
    for name, digest in manifest['files'].items():
        target = (path / name).resolve()
        if not target.is_relative_to(path) or not target.is_file() or file_hash(target) != digest:
            raise ValueError(f'Invalid or corrupted bundle file: {name}')
    return manifest

def find_data(input_root=Path('/kaggle/input')):
    for path in (input_root / 'rsna-knee-abnormality-detection', input_root / 'competitions/rsna-knee-abnormality-detection'):
        if (path / 'test.csv').is_file():
            return path
    raise FileNotFoundError('Attach the RSNA Knee competition data.')

def find_bundle(input_root=Path('/kaggle/input')):
    paths = list(input_root.glob('*/run/bundle/assets.json'))
    paths += list(input_root.glob('notebooks/*/*/run/bundle/assets.json'))
    paths += list(input_root.glob('kernels/*/*/run/bundle/assets.json'))
    if len(paths) != 1:
        raise ValueError('Attach exactly one completed experiment training output.')
    return paths[0].parent

def make_transform(config):
    from torchvision.transforms import InterpolationMode
    from torchvision.transforms._presets import ImageClassification
    return ImageClassification(crop_size=config['crop'], resize_size=config['resize'], interpolation=InterpolationMode(config['interpolation']), antialias=True)

def ordered_slices(folder):
    import pydicom
    paths = sorted(folder.glob('*.dcm'))
    if not paths:
        raise FileNotFoundError(f'No DICOM slices in {folder}')
    headers = [pydicom.dcmread(p, stop_before_pixels=True, specific_tags=['ImagePositionPatient', 'ImageOrientationPatient', 'InstanceNumber']) for p in paths]
    if all((hasattr(h, 'ImagePositionPatient') and hasattr(h, 'ImageOrientationPatient') for h in headers)):
        orientation = np.asarray(headers[0].ImageOrientationPatient, dtype=float)
        normal = np.cross(orientation[:3], orientation[3:])
        if np.linalg.norm(normal) <= 0.9:
            raise ValueError('Invalid DICOM orientation.')
        coordinates = np.asarray([np.dot(np.asarray(h.ImagePositionPatient, dtype=float), normal) for h in headers])
    elif all((hasattr(h, 'InstanceNumber') for h in headers)):
        coordinates = np.asarray([float(h.InstanceNumber) for h in headers])
    else:
        raise ValueError(f'No slice position or InstanceNumber: {folder}')
    if not np.isfinite(coordinates).all():
        raise ValueError('Non-finite slice positions.')
    order = np.argsort(coordinates, kind='stable')
    coordinates = coordinates[order]
    span = np.ptp(coordinates)
    positions = (coordinates - coordinates.min()) / span if span > 0 else np.full(len(paths), 0.5)
    return ([paths[i] for i in order], positions.astype(np.float32))

def image_tensor(path, transform):
    import pydicom
    ds = pydicom.dcmread(path)
    pixels = ds.pixel_array.astype(np.float32)
    if pixels.ndim != 2 or not np.isfinite(pixels).all():
        raise ValueError(f'Invalid MRI slice: {path}')
    pixels = pixels * float(getattr(ds, 'RescaleSlope', 1)) + float(getattr(ds, 'RescaleIntercept', 0))
    if getattr(ds, 'PhotometricInterpretation', '') == 'MONOCHROME1':
        pixels = -pixels
    low, high = np.percentile(pixels, [1, 99])
    if high <= low:
        low, high = (float(pixels.min()), float(pixels.max()))
    if high <= low:
        raise ValueError(f'Constant MRI slice: {path}')
    pixels = np.clip((pixels - low) / (high - low), 0, 1)
    return transform(torch.from_numpy(pixels).float().unsqueeze(0).repeat(3, 1, 1))

@torch.inference_mode()
def study_tokens(uid, series, image_root, encoder, transform, config, device):
    count, dim = (config['slices'], config['feature_dim'])
    features = torch.zeros(3, count, dim)
    mask = torch.zeros(3, count, dtype=torch.bool)
    positions = torch.zeros(3, count)
    rows = series.loc[series.StudyInstanceUID == uid]
    for plane_index, plane in enumerate(PLANES):
        choices = rows.loc[rows.Anatomical_Plane == plane].sort_values(['Fluid_Sensitive', 'Fat_Suppression', 'SeriesInstanceUID'], ascending=[False, False, True])
        if choices.empty:
            continue
        paths, coords = ordered_slices(image_root / uid / choices.iloc[0].SeriesInstanceUID)
        indices = np.unique(np.rint(np.linspace(0.25, 0.75, count) * (len(paths) - 1)).astype(int))
        batch = torch.stack([image_tensor(paths[i], transform) for i in indices]).to(device)
        output = encoder(batch).cpu().float()
        if output.shape != (len(indices), dim) or not torch.isfinite(output).all():
            raise ValueError('Unexpected encoder features.')
        features[plane_index, :len(indices)] = output
        mask[plane_index, :len(indices)] = True
        positions[plane_index, :len(indices)] = torch.from_numpy(coords[indices])
    if not mask.any():
        raise ValueError(f'No supported MRI planes for study {uid}')
    return (features, mask, positions)

def pooled_features(tokens, mask):
    clean = torch.where(mask[..., None], tokens, 0.0)
    return (clean.sum(2) / mask.sum(2).clamp_min(1)[..., None]).flatten(1)

class StudyHead(nn.Module):

    def __init__(self, config, mean, std):
        super().__init__()
        self.mode = config['aggregation']
        self.position = config['position']
        self.register_buffer('mean', mean.clone())
        self.register_buffer('std', std.clone())
        if self.mode == 'mean':
            self.classifier = nn.Linear(3 * config['feature_dim'], 12)
        elif self.mode == 'attention':
            self.project = nn.Linear(config['feature_dim'], 64)
            layer = nn.TransformerEncoderLayer(64, 4, 128, dropout=0.0, batch_first=True, norm_first=True)
            self.attention = nn.TransformerEncoder(layer, 1, enable_nested_tensor=False)
            self.classifier = nn.Linear(3 * 64, 12)
            self.plane_embedding = nn.Parameter(torch.zeros(3, 64)) if config['plane_embedding'] else None
            self.register_buffer('frequencies', torch.exp(torch.arange(0, 64, 2) * (-math.log(10000.0) / 64)))
        else:
            raise ValueError(f'Unknown aggregation: {self.mode}')

    def forward(self, tokens, mask, positions):
        if self.mode == 'mean':
            return self.classifier((pooled_features(tokens, mask) - self.mean) / self.std)
        clean = torch.where(mask[..., None], tokens, 0.0)
        x = self.project((clean - self.mean) / self.std)
        if self.position:
            angles = positions[..., None] * (2 * math.pi) * self.frequencies
            x = x + torch.stack((angles.sin(), angles.cos()), dim=-1).flatten(-2)
        if self.plane_embedding is not None:
            x = x + self.plane_embedding[None, :, None, :]
        b, planes, slices, dim = x.shape
        flat_mask = mask.reshape(b * planes, slices)
        safe_mask = flat_mask.clone()
        safe_mask[~safe_mask.any(1), 0] = True
        x = self.attention(x.reshape(b * planes, slices, dim), src_key_padding_mask=~safe_mask)
        x = torch.where(flat_mask[..., None], x, 0.0)
        vectors = x.sum(1) / flat_mask.sum(1).clamp_min(1)[:, None]
        return self.classifier(vectors.reshape(b, planes * dim))

def make_folds(ids, y):
    from sklearn.model_selection import KFold
    if len(set(ids)) != len(ids) or len(y) != len(ids) or (not np.isin(y, [0, 1]).all()):
        raise ValueError('Expected unique IDs and complete binary labels.')
    order = np.argsort(ids)
    folds = np.full(len(ids), -1, dtype=int)
    for fold, (train, valid) in enumerate(KFold(3, shuffle=True, random_state=42).split(order)):
        for index in (order[train], order[valid]):
            count = y[index].sum(0)
            if not ((count > 0) & (count < len(index))).all():
                raise ValueError('Each fold/label needs positives and negatives.')
        folds[order[valid]] = fold
    return folds

def metric_scores(y, logits):
    from sklearn.metrics import roc_auc_score
    probabilities = logits.sigmoid().numpy()
    aucs = roc_auc_score(y, probabilities, average=None)
    return {'macro_auc': float(np.mean(aucs)), 'per_label_auc': dict(zip(LABELS, aucs.tolist())), 'bce': nn.functional.binary_cross_entropy_with_logits(logits, torch.from_numpy(y)).item()}

def save_submission(ids, probabilities, sample_columns, target):
    probabilities = np.asarray(probabilities)
    if not ids or len(set(ids)) != len(ids) or list(sample_columns) != ['StudyInstanceUID', *LABELS] or (probabilities.shape != (len(ids), 12)) or (not np.isfinite(probabilities).all()) or (not ((probabilities >= 0) & (probabilities <= 1)).all()):
        raise ValueError('Invalid submission IDs, label order, shape or probabilities.')
    frame = pd.DataFrame(probabilities, columns=LABELS)
    frame.insert(0, 'StudyInstanceUID', ids)
    target = Path(target)
    temporary = target.with_suffix('.csv.part')
    frame.to_csv(temporary, index=False)
    temporary.replace(target)
    return frame


In [ ]:
"""Only the selected constructor is embedded in a training notebook."""
import torch
from torch import nn

def resnet18_encoder(pretrained=True):
    from torchvision.models import ResNet18_Weights, resnet18
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
    model.fc = nn.Identity()
    return model

def dinov2_encoder(pretrained=True):
    import os
    os.environ['XFORMERS_DISABLED'] = '1'
    return torch.hub.load('facebookresearch/dinov2:7764ea0f912e53c92e82eb78a2a1631e92725fc8', 'dinov2_vits14', pretrained=pretrained, trust_repo=True, verbose=False)

def swin_t_encoder(pretrained=True):
    from torchvision.models import Swin_T_Weights, swin_t
    model = swin_t(weights=Swin_T_Weights.IMAGENET1K_V1 if pretrained else None)
    model.head = nn.Identity()
    return model

def convnext_tiny_encoder(pretrained=True):
    from torchvision.models import ConvNeXt_Tiny_Weights, convnext_tiny
    model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if pretrained else None)
    model.classifier[2] = nn.Identity()
    return model

def vit_b16_encoder(pretrained=True):
    from torchvision.models import ViT_B_16_Weights, vit_b_16
    model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1 if pretrained else None)
    model.heads = nn.Identity()
    return model


In [ ]:
"""Report-supervised MRI experiments; shared verbatim by train and offline submit."""
import re

def make_targets(train, labels, config):
    train = train.sort_values('StudyInstanceUID').reset_index(drop=True).copy()
    if train.StudyInstanceUID.isna().any() or not train.StudyInstanceUID.is_unique or labels.StudyInstanceUID.isna().any() or (not labels.StudyInstanceUID.is_unique) or (not set(labels.StudyInstanceUID) <= set(train.StudyInstanceUID)):
        raise ValueError('Invalid, duplicate or foreign study IDs in label source.')
    required = [x + suffix for x in LABELS for suffix in ('', '__conf', '__verdict')]
    if not set(required) <= set(labels.columns):
        raise ValueError('Missing report label columns.')
    values = labels[[*LABELS, *[x + '__conf' for x in LABELS]]].to_numpy(dtype=np.float32)
    if not np.isfinite(values).all() or not ((values >= 0) & (values <= 1)).all():
        raise ValueError('Weak scores/confidences must be finite values in [0,1].')
    if not labels[[x + '__verdict' for x in LABELS]].isin(['YES', 'NO', 'UNK']).all().all():
        raise ValueError('Unknown report verdict.')
    observed = train[LABELS].notna()
    if (observed.any(axis=1) != observed.all(axis=1)).any():
        raise ValueError('Unexpected partially gold-labelled study.')
    gold = observed.all(axis=1).to_numpy()
    gold_y = train.loc[gold, LABELS].to_numpy(dtype=np.float32)
    if not np.isin(gold_y, [0, 1]).all():
        raise ValueError('Gold targets must be binary.')
    source = labels.set_index('StudyInstanceUID').reindex(train.StudyInstanceUID)
    y = source[LABELS].fillna(0.5).to_numpy(dtype=np.float32)
    conf = source[[x + '__conf' for x in LABELS]].fillna(0).to_numpy(dtype=np.float32)
    known = source[[x + '__verdict' for x in LABELS]].isin(['YES', 'NO']).to_numpy()
    w = ((0.25 + 0.75 * conf) * known).astype(np.float32)
    y[gold], w[gold] = (gold_y, 3.0)
    reports = train.Report.fillna('').map(lambda s: re.sub('\\s+', ' ', str(s)).strip().casefold())
    if reports.eq('').any():
        raise ValueError('Missing report: cannot audit report-group leakage.')
    groups = reports.map(lambda s: hashlib.sha256(s.encode()).hexdigest()).to_numpy()
    if len(set(groups[gold])) != int(gold.sum()):
        raise ValueError('Duplicate gold reports require grouped gold CV.')
    overlap = ~gold & np.isin(groups, groups[gold])
    keep = gold | config['weak_labels'] & ~overlap & (w.sum(1) > 0)
    audit = {'gold_studies': int(gold.sum()), 'weak_studies': int((keep & ~gold).sum()), 'weak_gold_report_overlap': int(overlap.sum()), 'weak_without_known_targets': int((~gold & (w.sum(1) == 0)).sum()), 'missing_source_rows': int(source.ACL.isna().sum()), 'known_weak_targets': {name: int(((w[:, i] > 0) & keep & ~gold).sum()) for i, name in enumerate(LABELS)}, 'unknown_policy': 'zero loss weight; never assume a negative', 'teacher_gold_independence': 'not established; development CV only'}
    if 'weak_min_confidence' in config:
        threshold = config['weak_min_confidence']
        if not 0 <= threshold <= 1:
            raise ValueError('Invalid confidence threshold.')
        removed = keep[:, None] & ~gold[:, None] & known & (conf < threshold)
        w[removed] = 0.0
        verdicts = source[[x + '__verdict' for x in LABELS]].to_numpy()
        audit['confidence_filter'] = dict(threshold=threshold, removed_yes={k: int((removed[:, i] & (verdicts[:, i] == 'YES')).sum()) for i, k in enumerate(LABELS)}, removed_no={k: int((removed[:, i] & (verdicts[:, i] == 'NO')).sum()) for i, k in enumerate(LABELS)}, remaining_weak_weight={k: float(w[keep & ~gold, i].sum()) for i, k in enumerate(LABELS)}, zero_supervision_weak_studies=int((keep & ~gold & (w.sum(1) == 0)).sum()))
    return (train.loc[keep].reset_index(drop=True), y[keep], w[keep], gold[keep], groups[keep], audit)

def find_report_labels(config, input_root=Path('/kaggle/input')):
    paths = [input_root / 'rsna-knee-llm-labels/report_labels_v2.csv', input_root / 'datasets/pilkwang/rsna-knee-llm-labels/report_labels_v2.csv']
    found = [p for p in paths if p.is_file()]
    if len(found) != 1 or file_hash(found[0]) != config['labels_sha256']:
        raise ValueError('Attach the pinned Pilkwang report label dataset; CSV hash must match.')
    return found[0]

def find_training_run(experiment_id, input_root=Path('/kaggle/input')):
    slug = 'rsna-knee-' + experiment_id.replace('_', '-') + '-train'
    paths = [input_root / slug / 'run', input_root / 'notebooks/choiyoungwhan' / slug / 'run', input_root / 'kernels/choiyoungwhan' / slug / 'run']
    found = [p for p in paths if (p / 'bundle/assets.json').is_file()]
    if len(found) != 1:
        raise ValueError(f'Attach exactly the pinned training output of {experiment_id}.')
    meta = json.loads((found[0] / 'bundle/assets.json').read_text())
    if meta['experiment']['id'] != experiment_id:
        raise ValueError('Dependency experiment mismatch.')
    return found[0]

def weighted_bce(logits, targets, weights):
    loss = torch.nn.functional.binary_cross_entropy_with_logits(logits.float(), targets.float(), reduction='none')
    return (loss * weights).sum() / weights.sum().clamp_min(1e-08)

def per_label_weighted_bce(logits, targets, weights):
    loss = torch.nn.functional.binary_cross_entropy_with_logits(logits.float(), targets.float(), reduction='none')
    denominator = weights.sum(0)
    valid = denominator > 0
    values = (loss * weights).sum(0) / denominator.clamp_min(1e-08)
    return (values * valid).sum() / valid.sum().clamp_min(1)

def validate_weak_bundle(path, config):
    path = Path(path).resolve()
    manifest = json.loads((path / 'assets.json').read_text())
    if manifest['experiment'] != config or manifest['labels'] != LABELS:
        raise ValueError('Wrong weak experiment bundle.')
    required = {f'members/{eid}/assets.json' for eid in config['train_dependencies']} if config.get('ensemble') else {'encoder.ts', 'model.pt'}
    files = manifest['files']
    wheels = {p.relative_to(path).as_posix() for p in (path / 'wheels').glob('*.whl')}
    if not required <= files.keys() or len(wheels) != 2 or (not wheels <= files.keys()) or (not any((x.startswith('wheels/pydicom-3.0.2-') for x in wheels))) or (not any((x.startswith('wheels/python_gdcm-3.2.6-') for x in wheels))):
        raise ValueError('Missing weights or offline DICOM dependencies.')
    for name, digest in files.items():
        target = (path / name).resolve()
        if not target.is_relative_to(path) or not target.is_file() or file_hash(target) != digest:
            raise ValueError(f'Invalid bundle artifact: {name}')
    if config.get('ensemble'):
        for eid in config['train_dependencies']:
            child = (path / 'members' / eid).resolve()
            if not child.is_relative_to(path):
                raise ValueError('Invalid member path.')
            child_config = json.loads((child / 'assets.json').read_text())['experiment']
            if child_config['id'] != eid:
                raise ValueError('Wrong ensemble member.')
            validate_weak_bundle(child, child_config)
    return manifest


In [ ]:
"""Slice tokens with explicit coordinates; identical training and inference path."""

def spatial_order(folder):
    import pydicom
    paths = sorted(folder.glob('*.dcm'))
    if not paths:
        raise FileNotFoundError(folder)
    headers = [pydicom.dcmread(p, stop_before_pixels=True, specific_tags=['ImagePositionPatient', 'ImageOrientationPatient', 'InstanceNumber']) for p in paths]
    physical = all((hasattr(h, 'ImagePositionPatient') and hasattr(h, 'ImageOrientationPatient') for h in headers))
    if physical:
        orientation = np.asarray(headers[0].ImageOrientationPatient, dtype=float)
        if any((not np.allclose(h.ImageOrientationPatient, orientation, atol=0.0001) for h in headers)):
            raise ValueError('Inconsistent orientations inside one series.')
        normal = np.cross(orientation[:3], orientation[3:])
        if not 0.99 < np.linalg.norm(normal) < 1.01:
            raise ValueError('Invalid DICOM orientation.')
        values = np.array([np.dot(h.ImagePositionPatient, normal) for h in headers])
    elif all((hasattr(h, 'InstanceNumber') for h in headers)):
        values = np.array([float(h.InstanceNumber) for h in headers])
    else:
        raise ValueError('Missing physical and instance coordinates.')
    if not np.isfinite(values).all():
        raise ValueError('Non-finite DICOM coordinates.')
    order = np.argsort(values, kind='stable')
    coords = np.zeros((len(paths), 3), dtype=np.float32)
    coords[:, 0] = np.arange(len(paths))
    coords[:, 1] = values[order] - values[order][0] if physical else 0.0
    coords[:, 2] = float(physical)
    return ([paths[i] for i in order], coords)

def selected_windows(length, config):
    if length < 1 or config['neighbors'] not in (3, 5, 10) or config['centers'] < 1:
        raise ValueError('Invalid slice selection.')
    start, end = config.get('center_range', [0.25, 0.75])
    if not 0 <= start < end <= 1:
        raise ValueError('Invalid center range.')
    centers = np.unique(np.rint(np.linspace(start, end, config['centers']) * (length - 1)).astype(int))
    if config['representation'] == 'triplet':
        if config['neighbors'] != 3:
            raise ValueError('Triplet encoder requires three channels.')
        return [tuple(np.clip([c - 1, c, c + 1], 0, length - 1)) for c in centers]
    if config['representation'] != 'single':
        raise ValueError('Unknown representation.')
    left = (config['neighbors'] - 1) // 2
    indices = {int(np.clip(c + j, 0, length - 1)) for c in centers for j in range(-left, config['neighbors'] - left)}
    return [(i,) for i in sorted(indices)]

def spatial_views(uid, series, image_root, transform, config, audit=None):
    count = config['centers'] * (config['neighbors'] if config['representation'] == 'single' else 1)
    images = torch.zeros(6, count, 3, config['crop'], config['crop'])
    mask = torch.zeros(6, count, dtype=torch.bool)
    coords = torch.zeros(6, count, 3)
    rows = series.loc[series.StudyInstanceUID == uid]
    for slot, (plane, fs) in enumerate(((p, f) for p in PLANES for f in (True, False))):
        choices = rows.loc[(rows.Anatomical_Plane == plane) & (rows.Fat_Suppression == fs)]
        choices = choices.sort_values(['Fluid_Sensitive', 'SeriesInstanceUID'], ascending=[False, True])
        if choices.empty:
            continue
        paths, positions = spatial_order(image_root / uid / choices.iloc[0].SeriesInstanceUID)
        decoded = {}
        windows = selected_windows(len(paths), config)
        if audit is not None:
            indices = sorted({i for window in windows for i in window})
            audit.append(dict(StudyInstanceUID=uid, slot=slot, plane=plane, fat_suppression=fs, original_slices=len(paths), unique_selected_slices=len(indices), tokens=len(windows), selected_index_min=min(indices), selected_index_max=max(indices), valid_fraction=len(windows) / count))
        for j, window in enumerate(windows):
            for index in window:
                if index not in decoded:
                    decoded[index] = image_tensor(paths[index], lambda x: x)[0]
            image = torch.stack([decoded[i] for i in window]) if len(window) == 3 else decoded[window[0]][None].repeat(3, 1, 1)
            images[slot, j] = transform(image)
            coords[slot, j] = torch.from_numpy(positions[window[len(window) // 2]])
            mask[slot, j] = True
    if not mask.any():
        raise ValueError(f'No supported images for {uid}')
    return (images, mask, coords)

def spatial_signature(config):
    signature = {k: config[k] for k in ('representation', 'neighbors', 'centers', 'crop', 'resize', 'interpolation', 'pipeline_sha256')}
    signature['pipeline_sha256'] = config.get('feature_pipeline_sha256', config['pipeline_sha256'])
    if list(config.get('center_range', [0.25, 0.75])) != [0.25, 0.75]:
        signature['center_range'] = list(config['center_range'])
    return signature

@torch.inference_mode()
def spatial_tokens(uid, series, image_root, encoder, transform, config, device, audit=None):
    images, mask, coords = spatial_views(uid, series, image_root, transform, config, audit=audit)
    output = torch.cat([encoder(x.to(device)).cpu().float() for x in images[mask].split(16)])
    if output.shape != (int(mask.sum()), config['feature_dim']) or not torch.isfinite(output).all():
        raise ValueError('Invalid spatial features.')
    tokens = torch.zeros(*mask.shape, config['feature_dim'], dtype=torch.float16)
    tokens[mask] = output.half()
    return (tokens, mask, coords)

def rotary(x, position):
    dim = x.shape[-1]
    if dim % 2:
        raise ValueError('Rotary head dimension must be even.')
    frequencies = torch.exp(torch.arange(0, dim, 2, device=x.device, dtype=torch.float32) * (-math.log(10000.0) / dim))
    angle = position.float()[:, None, :, None] * frequencies
    a, b = (x.float()[..., 0::2], x.float()[..., 1::2])
    return torch.stack((a * angle.cos() - b * angle.sin(), a * angle.sin() + b * angle.cos()), dim=-1).flatten(-2).to(x.dtype)

class SliceBlock(nn.Module):

    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.norm1, self.norm2 = (nn.LayerNorm(dim), nn.LayerNorm(dim))
        self.qkv, self.out = (nn.Linear(dim, 3 * dim), nn.Linear(dim, dim))
        self.ffn = nn.Sequential(nn.Linear(dim, 2 * dim), nn.GELU(), nn.Linear(2 * dim, dim))

    def forward(self, x, mask, position, use_rope):
        b, t, d = x.shape
        q, k, v = self.qkv(self.norm1(x)).reshape(b, t, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4).unbind(0)
        if use_rope:
            q, k = (rotary(q, position), rotary(k, position))
        scores = q.float() @ k.float().transpose(-1, -2) / math.sqrt(d // self.heads)
        scores = scores.masked_fill(~mask[:, None, None, :], -10000.0).softmax(-1)
        context = (scores.to(v.dtype) @ v).transpose(1, 2).reshape(b, t, d)
        x = x + self.out(context)
        return (x + self.ffn(self.norm2(x))) * mask[..., None]

class SpatialHead(nn.Module):

    def __init__(self, config, mean, std):
        super().__init__()
        self.config = config
        d = config['hidden_dim']
        if d % config['heads'] or d // config['heads'] % 2:
            raise ValueError('Invalid attention dimensions.')
        self.register_buffer('mean', mean.float().clone())
        self.register_buffer('std', std.float().clone())
        self.project = nn.Linear(config['feature_dim'], d)
        self.pool_score = nn.Sequential(nn.Linear(d, 64), nn.Tanh(), nn.Linear(64, 12))
        self.class_weight = nn.Parameter(torch.empty(12, 6 * d))
        self.class_bias = nn.Parameter(torch.zeros(12))
        nn.init.normal_(self.class_weight, std=0.01)
        self.coord_kind = nn.Embedding(2, d)
        nn.init.zeros_(self.coord_kind.weight)
        if config['sequence']:
            self.slice_block = SliceBlock(d, config['heads'])
        if config['plane_embedding']:
            self.plane = nn.Embedding(3, d)
            self.protocol = nn.Embedding(2, d)
        mode = config['fusion']
        if mode == 'mlp':
            self.mlp = nn.Sequential(nn.Linear(6 * d, d), nn.GELU(), nn.Linear(d, 1))
        elif mode == 'gate':
            self.gate = nn.Linear(d, 1)
            self.gate_weight = nn.Parameter(torch.randn(12, d) * 0.01)
        elif mode == 'cross':
            self.queries = nn.Parameter(torch.randn(12, d) * 0.02)
            self.memory_norm, self.query_norm = (nn.LayerNorm(d), nn.LayerNorm(d))
            self.cross = nn.MultiheadAttention(d, config['heads'], batch_first=True, dropout=0.0)
            self.read_norm = nn.LayerNorm(d)
            self.read_ffn = nn.Sequential(nn.Linear(d, 2 * d), nn.GELU(), nn.Linear(2 * d, d))
            self.read_weight = nn.Parameter(torch.randn(12, d) * 0.01)
        elif mode != 'concat':
            raise ValueError('Unsupported fusion.')

    def forward(self, tokens, mask, coords):
        if tokens.ndim != 4 or tokens.shape[1] != 6 or mask.shape != tokens.shape[:3] or (coords.shape != (*mask.shape, 3)):
            raise ValueError('Invalid token/coordinate shape.')
        if not mask.flatten(1).any(1).all():
            raise ValueError('Each study requires a valid image.')
        coords = torch.where(mask[..., None], coords, 0.0)
        clean = torch.where(mask[..., None], tokens.float(), self.mean)
        if not torch.isfinite(coords).all() or not torch.isfinite(clean).all():
            raise ValueError('Non-finite valid token or coordinate.')
        x = self.project((clean - self.mean) / self.std)
        x = x + self.coord_kind((coords[..., 2] > 0.5).long())
        b, s, t, d = x.shape
        if self.config['plane_embedding']:
            slot = torch.arange(6, device=x.device)
            x = x + (self.plane(slot // 2) + self.protocol(slot % 2))[None, :, None, :]
        pos = coords[..., 0]
        if self.config['position'] == 'rope_mm':
            pos = torch.where(coords[..., 2] > 0.5, coords[..., 1] / self.config['mm_unit'], pos)
        if self.config['position'] == 'additive':
            freq = torch.exp(torch.arange(0, d, 2, device=x.device) * (-math.log(10000.0) / d))
            angles = pos[..., None] * freq
            x = x + torch.stack((angles.sin(), angles.cos()), -1).flatten(-2)
        x = x * mask[..., None]
        if self.config['sequence']:
            flat = x.reshape(b * s, t, d)
            flat_mask = mask.reshape(b * s, t)
            active = flat_mask.any(1)
            result = torch.zeros_like(flat)
            result[active] = self.slice_block(flat[active], flat_mask[active], pos.reshape(b * s, t)[active], self.config['position'].startswith('rope'))
            x = result.reshape(b, s, t, d)
        if self.config['fusion'] == 'cross':
            memory = self.memory_norm(x.flatten(1, 2))
            query = self.queries[None].expand(b, -1, -1)
            read, _ = self.cross(self.query_norm(query), memory, memory, key_padding_mask=~mask.flatten(1), need_weights=False)
            read = query + read
            read = read + self.read_ffn(self.read_norm(read))
            return (read * self.read_weight).sum(-1) + self.class_bias
        attention = self.pool_score(x).masked_fill(~mask[..., None], -10000.0).softmax(2) * mask[..., None]
        pooled = torch.einsum('bstc,bstd->bscd', attention, x).permute(0, 2, 1, 3)
        if self.config['fusion'] == 'mlp':
            return self.mlp(pooled.flatten(2)).squeeze(-1) + self.class_bias
        if self.config['fusion'] == 'gate':
            weight = self.gate(pooled).squeeze(-1).masked_fill(~mask.any(2)[:, None, :], -10000.0).softmax(-1)
            return ((pooled * weight[..., None]).sum(2) * self.gate_weight).sum(-1) + self.class_bias
        return (pooled.flatten(2) * self.class_weight).sum(-1) + self.class_bias


In [ ]:
"""Training entry point; embedded after the core and one backbone constructor."""
import importlib.metadata
import platform
import subprocess
import sys
import time

def export_encoder(encoder, config, target):
    torch.backends.mha.set_fastpath_enabled(False)
    encoder = encoder.cpu().eval().requires_grad_(False)
    size = config['crop']
    example = torch.zeros(1, 3, size, size)
    with torch.no_grad():
        traced = torch.jit.trace(encoder, example, check_trace=False)
        for batch_size in range(1, config['slices'] + 1):
            probe = torch.rand(batch_size, 3, size, size)
            actual, expected = (traced(probe), encoder(probe))
            print(f'Encoder export batch={batch_size}: max_abs_error={(actual - expected).abs().max().item():.8g}', flush=True)
            torch.testing.assert_close(actual, expected, rtol=0.0001, atol=0.0001)
    traced.save(str(target))
    loaded = torch.jit.load(str(target), map_location='cpu').eval()
    with torch.no_grad():
        torch.testing.assert_close(loaded(probe), encoder(probe), rtol=0.0001, atol=0.0001)
    return loaded


In [ ]:
"""Train report-supervised ablations and save fully offline inference bundles."""
import shutil
import subprocess
import sys
import time

def write_weak_metrics(config, runs, out):
    metrics = {'experiment': config, 'runs': runs, 'macro_auc_mean': float(np.mean([r['macro_auc'] for r in runs])), 'macro_auc_seed_std': float(np.std([r['macro_auc'] for r in runs])), 'fold_auc_mean': float(np.mean([f['macro_auc'] for r in runs for f in r['folds']])), 'bce_mean': float(np.mean([r['bce'] for r in runs])), 'note': 'Gold58 development OOF; public teacher independence is not established. Seed SD is not a CI.'}
    (out / 'metrics.json').write_text(json.dumps(metrics, indent=2))
    return metrics

def freeze_swin_except_last_stage(encoder):
    encoder.eval().requires_grad_(False)
    encoder.features[-1].requires_grad_(True)
    encoder.norm.requires_grad_(True)
    return [p for p in encoder.parameters() if p.requires_grad]

def write_weak_manifest(config, bundle, table_count, gold_count, extra=None):
    manifest = {'experiment': config, 'labels': LABELS, 'python': list(sys.version_info[:2]), 'torch': torch.__version__, 'final_training_studies': table_count, 'gold_training_studies': gold_count, 'weak_training_studies': table_count - gold_count, 'files': {p.relative_to(bundle).as_posix(): file_hash(p) for p in sorted(bundle.rglob('*')) if p.is_file() and p != bundle / 'assets.json'}, **(extra or {})}
    (bundle / 'assets.json').write_text(json.dumps(manifest, indent=2))
    return manifest


In [ ]:
"""Train coordinate-aware heads and export offline bundles."""
import shutil
import subprocess
import sys
import time

def feature_statistics(tokens, mask, indices):
    total = torch.zeros(tokens.shape[-1], dtype=torch.float64)
    squares = torch.zeros_like(total)
    count = 0
    for batch in torch.as_tensor(indices, dtype=torch.long).split(32):
        values = tokens[batch][mask[batch]].double()
        if not torch.isfinite(values).all():
            raise ValueError('Invalid training features.')
        total += values.sum(0)
        squares += (values * values).sum(0)
        count += len(values)
    if count < 2:
        raise ValueError('Insufficient training features.')
    mean = total / count
    std = ((squares - total.square() / count) / (count - 1)).clamp_min(0).sqrt().clamp_min(0.01)
    return (mean.float(), std.float())

@torch.inference_mode()
def spatial_predict(head, tokens, mask, coords, index):
    device = next(head.parameters()).device
    return torch.cat([head(tokens[b].to(device), mask[b].to(device), coords[b].to(device)).cpu() for b in torch.as_tensor(index, dtype=torch.long).split(32)])

def weak_finetune_split(gold, groups):
    weak = np.flatnonzero(~gold)
    valid = weak[np.array([int(g[:8], 16) % 10 == 0 for g in groups[weak]])]
    train_index = np.setdiff1d(weak, valid)
    if not len(train_index) or not len(valid) or gold[train_index].any() or gold[valid].any():
        raise ValueError('Invalid FT split.')
    return (train_index, valid)

def compatible_spatial_cache(cache, config, encoder_hash, ids):
    return cache.get('view_signature') == spatial_signature(config) and cache.get('encoder_sha256') == encoder_hash and (cache.get('ids') == list(ids))


In [ ]:
"""Physical preprocessing and coarse spatial tokens shared by train and submit."""

def quadrant_pool(x):
    return torch.stack((x[:, :4, :4].mean((1, 2)), x[:, :4, 4:].mean((1, 2)), x[:, 4:, :4].mean((1, 2)), x[:, 4:, 4:].mean((1, 2))), 1)

def quadrant_mean(tokens):
    return (tokens * tokens.new_tensor([16 / 49, 12 / 49, 12 / 49, 9 / 49])[:, None]).sum(-2)

class QuadrantSwin(nn.Module):

    def __init__(self, pretrained=False):
        super().__init__()
        base = swin_t_encoder(pretrained=pretrained)
        self.features, self.norm = (base.features, base.norm)

    def forward(self, x):
        return quadrant_pool(self.norm(self.features(x)))

def physical_crop(pixels, spacing, field_mm, size):
    spacing = np.asarray(spacing, dtype=float)
    if spacing.shape != (2,) or not np.isfinite(spacing).all() or (spacing <= 0).any() or (not np.isfinite(field_mm)) or (field_mm <= 0) or (size < 2):
        raise ValueError('A physical crop requires valid positive PixelSpacing and field size.')
    h, w = pixels.shape
    if min(h, w) < 2:
        raise ValueError('MRI matrix is too small.')
    distance = torch.linspace(-field_mm / 2, field_mm / 2, size, device=pixels.device)
    gy, gx = torch.meshgrid(distance / (spacing[0] * (h - 1) / 2), distance / (spacing[1] * (w - 1) / 2), indexing='ij')
    grid = torch.stack((gx, gy), -1)[None].float()
    return torch.nn.functional.grid_sample(pixels[None, None].float(), grid, align_corners=True, padding_mode='zeros', mode='bilinear')[0, 0]

def canonical_orientation(pixels, orientation, plane):
    axes = {'Sagittal': ([0, 1, 0], [0, 0, -1]), 'Coronal': ([1, 0, 0], [0, 0, -1]), 'Axial': ([1, 0, 0], [0, 1, 0])}
    target_col, target_row = map(np.asarray, axes[plane])
    iop = np.asarray(orientation, dtype=float)
    if iop.shape != (6,) or not np.isfinite(iop).all() or (not np.allclose([np.linalg.norm(iop[:3]), np.linalg.norm(iop[3:])], 1.0, atol=0.01)) or (abs(np.dot(iop[:3], iop[3:])) > 0.01):
        raise ValueError('Invalid or missing ImageOrientationPatient.')
    col, row = (iop[:3], iop[3:])
    if abs(row @ target_col) + abs(col @ target_row) > abs(col @ target_col) + abs(row @ target_row):
        pixels = pixels.T
        col, row = (row, col)
    dots = np.array([col @ target_col, row @ target_row])
    if np.abs(dots).min() < 0.5:
        raise ValueError('IOP is inconsistent with the anatomical plane.')
    if dots[0] < 0:
        pixels = pixels.flip(1)
    if dots[1] < 0:
        pixels = pixels.flip(0)
    residual = float(np.degrees(np.arccos(np.clip(np.abs(dots), 0, 1))).max())
    return (pixels, residual)

def anatomy_image(path, plane, config, audit=None, flip=False):
    import pydicom
    pixels = image_tensor(path, lambda image: image)[0]
    dcm = pydicom.dcmread(path, stop_before_pixels=True, specific_tags=['PixelSpacing', 'ImageOrientationPatient'])
    residual = 0.0
    if config.get('field_mm'):
        pixels = physical_crop(pixels, getattr(dcm, 'PixelSpacing', []), config['field_mm'], config['crop'])
    if config.get('canonical'):
        pixels, residual = canonical_orientation(pixels, getattr(dcm, 'ImageOrientationPatient', []), plane)
    if flip:
        pixels = pixels.flip(1)
    image = pixels[None].repeat(3, 1, 1)
    if config.get('field_mm'):
        image = (image - image.new_tensor([0.485, 0.456, 0.406])[:, None, None]) / image.new_tensor([0.229, 0.224, 0.225])[:, None, None]
    else:
        image = make_transform(config)(image)
    if audit is not None:
        audit.append(dict(path=str(path), plane=plane, residual_oblique_degrees=residual))
    return image

def anatomy_views(uid, series, image_root, config, audit=None, flip=False, limit=None):
    count = config['centers'] * config['neighbors'] if limit is None else limit
    images = torch.zeros(6, count, 3, config['crop'], config['crop'])
    mask = torch.zeros(6, count, dtype=torch.bool)
    coords = torch.zeros(6, count, 3)
    rows = series.loc[series.StudyInstanceUID == uid]
    for slot, (plane, fs) in enumerate(((p, f) for p in PLANES for f in (True, False))):
        choices = rows.loc[(rows.Anatomical_Plane == plane) & (rows.Fat_Suppression == fs)]
        choices = choices.sort_values(['Fluid_Sensitive', 'SeriesInstanceUID'], ascending=[False, True])
        if choices.empty:
            continue
        paths, positions = spatial_order(image_root / uid / choices.iloc[0].SeriesInstanceUID)
        selected = [w[0] for w in selected_windows(len(paths), config)]
        if limit and len(selected) > limit:
            selected = [selected[i] for i in np.rint(np.linspace(0, len(selected) - 1, limit)).astype(int)]
        for j, i in enumerate(selected):
            images[slot, j] = anatomy_image(paths[i], plane, config, audit, flip)
            mask[slot, j] = True
            coords[slot, j] = torch.from_numpy(positions[i])
    if not mask.any():
        raise ValueError(f'No supported MRI series: {uid}')
    return (images, mask, coords)

@torch.inference_mode()
def anatomy_tokens(uid, series, image_root, encoder, config, device, audit=None, flip=False):
    images, mask, coords = anatomy_views(uid, series, image_root, config, audit, flip)
    values = torch.cat([encoder(x.to(device)).cpu().float() for x in images[mask].split(16)])
    if values.shape != (int(mask.sum()), 4, config['feature_dim']) or not torch.isfinite(values).all():
        raise ValueError('Invalid quadrant features.')
    tokens = torch.zeros(*mask.shape, 4, config['feature_dim'], dtype=torch.float16)
    tokens[mask] = values.half()
    return (tokens, mask, coords)

class AnatomyHead(SpatialHead):

    def forward(self, tokens, mask, coords):
        if tokens.ndim != 5 or tokens.shape[-2] != 4:
            raise ValueError('Expected four spatial regions per slice.')
        if not self.config['patch_attention']:
            return super().forward(quadrant_mean(tokens.float()), mask, coords)
        if mask.shape != tokens.shape[:3] or coords.shape != (*mask.shape, 3) or (not mask.flatten(1).any(1).all()):
            raise ValueError('Invalid masked study.')
        clean = torch.where(mask[..., None, None], tokens.float(), self.mean)
        coords = torch.where(mask[..., None], coords, 0.0)
        if not torch.isfinite(clean).all() or not torch.isfinite(coords).all():
            raise ValueError('Invalid valid feature.')
        x = self.project((clean - self.mean) / self.std)
        b, s, t, p, d = x.shape
        xy = x.new_tensor([[-1, -1], [1, -1], [-1, 1], [1, 1]])
        freq = torch.exp(torch.arange(0, d // 2, 2, device=x.device) * (-math.log(10000.0) / (d // 2)))
        angles = xy[:, :, None] * freq
        pe = torch.stack((angles.sin(), angles.cos()), -1).flatten(1)
        if pe.shape[-1] != d:
            raise ValueError('Patch position encoding requires hidden_dim divisible by four.')
        slot = torch.arange(6, device=x.device)
        x = x + pe[None, None, None] + self.coord_kind((coords[..., 2] > 0.5).long())[..., None, :]
        x = x + (self.plane(slot // 2) + self.protocol(slot % 2))[None, :, None, None, :]
        pos = torch.where(coords[..., 2] > 0.5, coords[..., 1] / self.config['mm_unit'], coords[..., 0])
        pos = pos.repeat_interleave(4, -1)
        expanded = mask.repeat_interleave(4, -1)
        x = x.flatten(2, 3) * expanded[..., None]
        flat = x.reshape(b * s, t * p, d)
        m = expanded.reshape(b * s, t * p)
        active = m.any(1)
        result = torch.zeros_like(flat)
        result[active] = self.slice_block(flat[active], m[active], pos.reshape(b * s, t * p)[active], True)
        memory = self.memory_norm(result.reshape(b, s * t * p, d))
        query = self.queries[None].expand(b, -1, -1)
        read, _ = self.cross(self.query_norm(query), memory, memory, key_padding_mask=~expanded.flatten(1), need_weights=False)
        read = query + read
        read = read + self.read_ffn(self.read_norm(read))
        return (read * self.read_weight).sum(-1) + self.class_bias

def reference_cdf(reference, values):
    reference, values = (np.asarray(reference), np.asarray(values))
    if reference.ndim != 2 or values.ndim != 2 or values.shape[1] != reference.shape[1] or (not len(reference)):
        raise ValueError('CDF requires a nonempty fixed training reference with matching classes.')
    if not np.isfinite(reference).all() or not np.isfinite(values).all():
        raise ValueError('Invalid CDF values.')
    result = np.empty_like(values, dtype=float)
    for c in range(values.shape[1]):
        sorted_values = np.sort(reference[:, c])
        rank = (np.searchsorted(sorted_values, values[:, c], 'left') + np.searchsorted(sorted_values, values[:, c], 'right')) / 2
        result[:, c] = (rank + 0.5) / (len(reference) + 1)
    return result


In [ ]:
"""Gold-held-out heads, saved fold models, fixed-reference rank fusion."""
import shutil
import time

def anatomy_statistics(tokens, mask, index):
    total = torch.zeros(tokens.shape[-1], dtype=torch.float64)
    squares = total.clone()
    n = 0
    for batch in torch.as_tensor(index).long().split(16):
        v = quadrant_mean(tokens[batch].double())[mask[batch]]
        if not torch.isfinite(v).all():
            raise ValueError('Invalid training features.')
        total += v.sum(0)
        squares += v.square().sum(0)
        n += len(v)
    if n < 2:
        raise ValueError('Insufficient training features.')
    return ((total / n).float(), ((squares - total.square() / n) / (n - 1)).clamp_min(0).sqrt().clamp_min(0.01).float())

def fit_anatomy_head(tokens, mask, coords, y, weights, index, seed, config):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    torch.manual_seed(seed)
    index = torch.as_tensor(index).long()
    head = AnatomyHead(config, *anatomy_statistics(tokens, mask, index)).to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    history = []
    for epoch in range(1, config['epochs'] + 1):
        head.train()
        numerator = denominator = 0.0
        steps = 0
        for batch in index[torch.randperm(len(index))].split(config['batch_size']):
            weight = weights[batch].sum().item()
            if weight == 0:
                continue
            optimizer.zero_grad(set_to_none=True)
            logits = head(tokens[batch].to(device), mask[batch].to(device), coords[batch].to(device))
            loss = weighted_bce(logits, y[batch].to(device), weights[batch].to(device))
            if not torch.isfinite(loss):
                raise ValueError('Non-finite head loss.')
            loss.backward()
            torch.nn.utils.clip_grad_norm_(head.parameters(), 5.0)
            optimizer.step()
            numerator += loss.item() * weight
            denominator += weight
            steps += 1
        if not steps:
            raise ValueError('No supervised batches.')
        history.append(dict(epoch=epoch, train_loss=numerator / denominator, optimizer_steps=steps))
    return (head.eval(), history)

def save_anatomy_head(head, config, path, **extra):
    torch.save(dict(state={k: v.detach().cpu() for k, v in head.state_dict().items()}, experiment=config, labels=LABELS, **extra), path)

def load_anatomy_head(path, config, device):
    saved = torch.load(path, map_location='cpu', weights_only=True)
    if saved['experiment'] != config or saved['labels'] != LABELS:
        raise ValueError('Wrong saved anatomy head.')
    head = AnatomyHead(config, saved['state']['mean'], saved['state']['std'])
    head.load_state_dict(saved['state'], strict=True)
    return head.to(device).eval()

def frame_metrics(frame, truth, config, out):
    runs = []
    for seed, rows in frame.groupby('seed'):
        y = truth.loc[rows.StudyInstanceUID, LABELS].to_numpy(dtype=np.float32)
        logits = torch.logit(torch.tensor(rows[LABELS].to_numpy(), dtype=torch.float32).clamp(1e-07, 1 - 1e-07))
        scores = [dict(fold=fold, **metric_scores(y[rows.fold.to_numpy() == fold], logits[rows.fold.to_numpy() == fold])) for fold in range(3)]
        runs.append(dict(seed=int(seed), **metric_scores(y, logits), folds=scores))
    frame.to_csv(out / 'oof_predictions.csv', index=False)
    return write_weak_metrics(config, runs, out)

def evaluate_anatomy_cv(tokens, mask, coords, y, w, gold, ids, config, out):
    gi, wi = (np.flatnonzero(gold), np.flatnonzero(~gold))
    gids = np.asarray(ids)[gold]
    folds = make_folds(gids.tolist(), y[gi].numpy())
    pd.DataFrame(dict(StudyInstanceUID=gids, fold=folds)).to_csv(out / 'folds.csv', index=False)
    (out / 'fold_models').mkdir()
    frames = []
    ranks = []
    history = []
    for seed in SEEDS:
        for fold in range(3):
            train_index = np.concatenate((wi, gi[folds != fold]))
            valid = gi[folds == fold]
            head, h = fit_anatomy_head(tokens, mask, coords, y, w, train_index, seed, config)
            probabilities = spatial_predict(head, tokens, mask, coords, valid).sigmoid().numpy()
            reference = spatial_predict(head, tokens, mask, coords, wi).sigmoid().numpy()
            frame = pd.DataFrame(probabilities, columns=LABELS)
            frame.insert(0, 'fold', fold)
            frame.insert(0, 'seed', seed)
            frame.insert(0, 'StudyInstanceUID', np.asarray(ids)[valid])
            frames.append(frame)
            ranked = frame.copy()
            ranked[LABELS] = reference_cdf(reference, probabilities)
            ranks.append(ranked)
            save_anatomy_head(head, config, out / 'fold_models' / f'{seed}_{fold}.pt', fold=fold, seed=seed, validation_ids=np.asarray(ids)[valid].tolist())
            history.extend((dict(seed=seed, fold=fold, **row) for row in h))
            print(f"{config['id']}: seed={seed} fold={fold} complete", flush=True)
            del head
    frame = pd.concat(frames, ignore_index=True)
    pd.concat(ranks, ignore_index=True).to_csv(out / 'oof_rank_predictions.csv', index=False)
    pd.DataFrame(history).to_csv(out / 'history.csv', index=False)
    truth = pd.DataFrame(y.numpy(), columns=LABELS, index=ids)
    return frame_metrics(frame, truth, config, out)

def anatomy_signature(config):
    return {k: config[k] for k in ('crop', 'resize', 'interpolation', 'neighbors', 'centers', 'representation', 'field_mm', 'canonical', 'pipeline_sha256')}

def checked_anatomy_run(eid):
    path = find_training_run(eid)
    cfg = json.loads((path / 'bundle/assets.json').read_text())['experiment']
    meta = validate_weak_bundle(path / 'bundle', cfg)
    return (path, cfg, meta)

def read_anatomy_cache(path, meta):
    if file_hash(path / 'tokens.pt') != meta['tokens_sha256']:
        raise ValueError('Corrupt cached features.')
    return torch.load(path / 'tokens.pt', map_location='cpu', weights_only=True)

def extract_anatomy_cache(ids, series, image_root, encoder, config, flip=False):
    count = config['centers'] * config['neighbors']
    n = len(ids)
    tokens = torch.zeros(n, 6, count, 4, config['feature_dim'], dtype=torch.float16)
    mask = torch.zeros(n, 6, count, dtype=torch.bool)
    coords = torch.zeros(n, 6, count, 3)
    audit = []
    started = time.monotonic()
    encoder = encoder.cuda().eval()
    for i, uid in enumerate(ids):
        if time.monotonic() - started > 7 * 3600:
            raise TimeoutError('Feature extraction exceeded 7h; no partial result accepted.')
        tokens[i], mask[i], coords[i] = anatomy_tokens(uid, series, image_root, encoder, config, 'cuda', audit, flip)
        if i % 50 == 0:
            print(f'features {i}/{n} elapsed={(time.monotonic() - started) / 60:.1f} min', flush=True)
    encoder.cpu()
    torch.cuda.empty_cache()
    return (dict(ids=list(ids), tokens=tokens, mask=mask, coords=coords), audit)

def blend_oof(frames):
    keys = ['StudyInstanceUID', 'seed', 'fold']
    ordered = [f.sort_values(['seed', 'StudyInstanceUID']).reset_index(drop=True) for f in frames]
    if not ordered or any((not ordered[0][keys].equals(f[keys]) for f in ordered[1:])):
        raise ValueError('Ensemble OOF patients/seeds/folds differ.')
    result = ordered[0].copy()
    result[LABELS] = np.mean([f[LABELS].to_numpy() for f in ordered], axis=0)
    return result

def train_anatomy_fusion(config, data, out):
    parents = [checked_anatomy_run(eid) for eid in config['train_dependencies']]
    paths = [p[0] for p in parents]
    if config['method'] == 'tta':
        path, cfg, meta = parents[0]
        if file_hash(path / 'oof_predictions.csv') != meta['oof_sha256']['oof_predictions.csv']:
            raise ValueError('Corrupt OOF predictions.')
        frame = pd.read_csv(path / 'oof_predictions.csv')
        ids = frame.StudyInstanceUID.drop_duplicates().tolist()
        encoder = torch.jit.load(str(path / 'bundle/encoder.ts'), map_location='cpu')
        cache, _ = extract_anatomy_cache(ids, pd.read_csv(data / 'train_series.csv'), data / 'train_series', encoder, cfg, flip=True)
        lookup = {uid: i for i, uid in enumerate(ids)}
        for (seed, fold), rows in frame.groupby(['seed', 'fold']):
            headpath = path / 'fold_models' / f'{seed}_{fold}.pt'
            if file_hash(headpath) != meta['fold_models_sha256'][headpath.name]:
                raise ValueError('Corrupt fold head.')
            saved = torch.load(headpath, map_location='cpu', weights_only=True)
            if set(saved['validation_ids']) != set(rows.StudyInstanceUID):
                raise ValueError('Wrong validation fold.')
            head = load_anatomy_head(headpath, cfg, 'cuda')
            indices = [lookup[uid] for uid in rows.StudyInstanceUID]
            p = spatial_predict(head, cache['tokens'], cache['mask'], cache['coords'], indices).sigmoid().numpy()
            frame.loc[rows.index, LABELS] = (rows[LABELS].to_numpy() + p) / 2
    else:
        filename = 'oof_rank_predictions.csv' if config['method'] == 'rank' else 'oof_predictions.csv'
        for path, cfg, meta in parents:
            if file_hash(path / filename) != meta['oof_sha256'][filename]:
                raise ValueError('Corrupt OOF predictions.')
        frame = blend_oof([pd.read_csv(p / filename) for p in paths])
    metrics = frame_metrics(frame, pd.read_csv(data / 'train.csv').set_index('StudyInstanceUID'), config, out)
    shutil.copy2(paths[0] / 'folds.csv', out / 'folds.csv')
    bundle = out / 'bundle'
    for path, cfg, meta in parents:
        shutil.copytree(path / 'bundle', bundle / 'members' / cfg['id'])
    shutil.copytree(paths[0] / 'bundle/wheels', bundle / 'wheels')
    write_weak_manifest(config, bundle, 4350, 58, dict(mixing=config['method'], members=config['train_dependencies']))
    return metrics

def train_anatomy_experiment(config, data, out):
    data, out = (Path(data), Path(out))
    if out.exists() and any(out.iterdir()):
        raise FileExistsError('Choose an empty output directory.')
    out.mkdir(parents=True, exist_ok=True)
    torch.set_num_threads(1)
    torch.manual_seed(42)
    torch.backends.mha.set_fastpath_enabled(False)
    if config.get('ensemble'):
        return train_anatomy_fusion(config, data, out)
    if not torch.cuda.is_available():
        raise RuntimeError('A Kaggle GPU is required.')
    table, y, w, gold, groups, audit = make_targets(pd.read_csv(data / 'train.csv'), pd.read_csv(find_report_labels(config)), config)
    if len(table) != 4350 or int(gold.sum()) != 58:
        raise ValueError('Training population changed.')
    y, w = (torch.from_numpy(y), torch.from_numpy(w))
    ids = table.StudyInstanceUID.tolist()
    (out / 'label_audit.json').write_text(json.dumps(audit, indent=2))
    parent, old, meta = checked_anatomy_run(config['train_dependencies'][0])
    bundle = out / 'bundle'
    bundle.mkdir()
    encoder = QuadrantSwin(False)
    old_encoder = torch.jit.load(str(parent / 'bundle/encoder.ts'), map_location='cpu')
    state = {k: v for k, v in old_encoder.state_dict().items() if k.startswith(('features.', 'norm.'))}
    encoder.load_state_dict(state, strict=True)
    del old_encoder
    if config.get('adapt_stage'):
        cp = find_adaptation(config['adapt_stage'])
        progress = json.loads((cp / 'progress.json').read_text())
        if progress['checkpoint_sha256'] != file_hash(cp / 'checkpoint.pt') or progress['recipe'] != adapt_recipe(config):
            raise ValueError('Wrong adaptation checkpoint.')
        saved = torch.load(cp / 'checkpoint.pt', map_location='cpu', weights_only=True)
        if saved['epoch'] != config['backbone_epochs'] or saved['segment'] != 3:
            raise ValueError('Incomplete adaptation epoch.')
        encoder.load_state_dict(saved['best_encoder'], strict=True)
        (out / 'backbone_audit.json').write_text(json.dumps(progress, indent=2))
        del saved
    encoder = export_encoder(encoder, dict(config, slices=16), bundle / 'encoder.ts')
    encoder_hash = file_hash(bundle / 'encoder.ts')
    started = time.monotonic()
    reused = False
    if old.get('architecture') == 'anatomy_v1' and anatomy_signature(old) == anatomy_signature(config) and (not config.get('adapt_stage')):
        cache = read_anatomy_cache(parent, meta)
        shutil.copy2(parent / 'bundle/encoder.ts', bundle / 'encoder.ts')
        encoder_hash = file_hash(bundle / 'encoder.ts')
        reused = True
        if cache['encoder_sha256'] != encoder_hash or cache['ids'] != ids or cache['view_signature'] != anatomy_signature(config):
            raise ValueError('Incompatible cached anatomy features.')
    else:
        cache, geometry = extract_anatomy_cache(ids, pd.read_csv(data / 'train_series.csv'), data / 'train_series', encoder, config)
        pd.DataFrame(geometry).to_csv(out / 'selection_audit.csv', index=False)
    cache.update(view_signature=anatomy_signature(config), encoder_sha256=encoder_hash)
    torch.save(cache, out / 'tokens.pt')
    del encoder
    tokens, mask, coords = (cache[k] for k in ('tokens', 'mask', 'coords'))
    timing = dict(feature_seconds=time.monotonic() - started, cached_studies=len(ids) if reused else 0, images=int(mask.sum()), gpu=torch.cuda.get_device_name(0))
    metrics = evaluate_anatomy_cv(tokens, mask, coords, y, w, gold, ids, config, out)
    head, history = fit_anatomy_head(tokens, mask, coords, y, w, np.arange(len(ids)), 42, config)
    save_anatomy_head(head, config, bundle / 'model.pt')
    reference = spatial_predict(head, tokens, mask, coords, np.flatnonzero(~gold)).sigmoid()
    torch.save(reference, bundle / 'reference.pt')
    pd.DataFrame(history).to_csv(out / 'final_fit_history.csv', index=False)
    shutil.copytree(parent / 'bundle/wheels', bundle / 'wheels')
    write_weak_manifest(config, bundle, 4350, 58, dict(tokens_sha256=file_hash(out / 'tokens.pt'), folds_sha256=file_hash(out / 'folds.csv'), head_parameters=sum((p.numel() for p in head.parameters())), label_audit=audit, fold_models_sha256={p.name: file_hash(p) for p in (out / 'fold_models').glob('*.pt')}, oof_sha256={name: file_hash(out / name) for name in ('oof_predictions.csv', 'oof_rank_predictions.csv')}))
    (out / 'timings.json').write_text(json.dumps(timing, indent=2))
    return metrics


In [ ]:
"""Weak-only full encoder adaptation, four bounded resumable jobs per epoch."""
import hashlib
import time

def adapt_recipe(config):
    keys = ('pipeline_sha256', 'labels_sha256')
    return hashlib.sha256(json.dumps({k: config[k] for k in keys}, sort_keys=True).encode()).hexdigest()

def segment_indices(gold, groups, epoch, segment):
    if epoch < 1 or segment not in range(4):
        raise ValueError('Invalid adaptation stage.')
    train_index, valid = weak_finetune_split(gold, groups)
    order = np.random.default_rng(42 + epoch).permutation(train_index)
    batches = [order[i:i + 8] for i in range(0, len(order), 8)]
    selected = np.array_split(np.arange(len(batches)), 4)[segment]
    return (np.concatenate([batches[i] for i in selected]) if len(selected) else np.array([], dtype=int), valid)

def validate_checkpoint(checkpoint, config, epoch, segment):
    prior = (epoch, segment - 1) if segment else (epoch - 1, 3)
    if checkpoint.get('recipe') != adapt_recipe(config) or (checkpoint.get('epoch'), checkpoint.get('segment')) != prior:
        raise ValueError('Checkpoint recipe or predecessor stage mismatch.')

def find_adaptation(stage, input_root=Path('/kaggle/input')):
    slug = 'rsna-knee-' + stage.replace('_', '-') + '-train'
    paths = [input_root / slug / 'adapt', input_root / 'notebooks/choiyoungwhan' / slug / 'adapt', input_root / 'kernels/choiyoungwhan' / slug / 'adapt']
    found = [p for p in paths if (p / 'progress.json').is_file()]
    if len(found) != 1:
        raise ValueError('Attach exactly the pinned adaptation stage ' + stage)
    return found[0]

def cpu_tree(value):
    if torch.is_tensor(value):
        return value.detach().cpu()
    if isinstance(value, dict):
        return {k: cpu_tree(v) for k, v in value.items()}
    if isinstance(value, list):
        return [cpu_tree(v) for v in value]
    if isinstance(value, tuple):
        return tuple((cpu_tree(v) for v in value))
    return value

def adapt_anatomy(config, epoch, segment, previous_stage, data, out):
    from torch.utils.checkpoint import checkpoint as recompute
    data, out = (Path(data), Path(out))
    if not torch.cuda.is_available():
        raise RuntimeError('Adaptation requires a GPU.')
    if out.exists() and any(out.iterdir()):
        raise FileExistsError('Output is not empty.')
    out.mkdir(parents=True, exist_ok=True)
    torch.set_num_threads(1)
    torch.manual_seed(42)
    torch.backends.mha.set_fastpath_enabled(False)
    table, y, w, gold, groups, audit = make_targets(pd.read_csv(data / 'train.csv'), pd.read_csv(find_report_labels(config)), config)
    if len(table) != 4350 or int(gold.sum()) != 58:
        raise ValueError('Training population changed.')
    train_index, valid = segment_indices(gold, groups, epoch, segment)
    if not len(train_index):
        raise ValueError('Empty adaptation segment.')
    y, w = (torch.from_numpy(y).cuda(), torch.from_numpy(w).cuda())
    series = pd.read_csv(data / 'train_series.csv')
    base, base_cfg, meta = checked_anatomy_run('u03_patch_attention')
    origin = file_hash(base / 'bundle/encoder.ts')
    encoder = QuadrantSwin(False)
    saved = None
    if previous_stage:
        folder = find_adaptation(previous_stage)
        progress = json.loads((folder / 'progress.json').read_text())
        if file_hash(folder / 'checkpoint.pt') != progress['checkpoint_sha256']:
            raise ValueError('Corrupt checkpoint.')
        saved = torch.load(folder / 'checkpoint.pt', map_location='cpu', weights_only=True)
        validate_checkpoint(saved, config, epoch, segment)
        if saved['origin_sha256'] != origin:
            raise ValueError('Adaptation origin changed.')
        encoder.load_state_dict(saved['encoder'])
        head = AnatomyHead(config, saved['head']['mean'], saved['head']['std'])
        head.load_state_dict(saved['head'])
    else:
        if (epoch, segment) != (1, 0):
            raise ValueError('Missing continuation checkpoint.')
        encoder.load_state_dict(torch.jit.load(str(base / 'bundle/encoder.ts'), map_location='cpu').state_dict())
        cache = read_anatomy_cache(base, meta)
        weak_train, _ = weak_finetune_split(gold, groups)
        head, _ = fit_anatomy_head(cache['tokens'], cache['mask'], cache['coords'], y.cpu(), w.cpu(), weak_train, 42, dict(config, epochs=1))
        del cache
    encoder = encoder.cuda().requires_grad_(True)
    head = head.cuda()
    optimizer = torch.optim.AdamW([dict(params=encoder.parameters(), lr=1e-05), dict(params=head.parameters(), lr=0.0001)], weight_decay=0.01)
    scaler = torch.amp.GradScaler('cuda')
    if saved:
        optimizer.load_state_dict(saved['optimizer'])
        scaler.load_state_dict(saved['scaler'])
        torch.set_rng_state(saved['rng'])
        torch.cuda.set_rng_state_all(saved['cuda_rng'])
    best = saved['best_score'] if saved else float('inf')
    best_encoder = saved['best_encoder'] if saved else None
    best_epoch = saved['best_epoch'] if saved else None
    history = saved['history'] if saved else []
    started = time.monotonic()
    deadline = started + 7 * 3600

    def predict(index, training):
        if time.monotonic() > deadline:
            raise TimeoutError('Adaptation segment exceeded 7h; resume previous completed segment.')
        images, mask, coords = anatomy_views(table.StudyInstanceUID.iloc[index], series, data / 'train_series', config, limit=8)
        with torch.autocast('cuda', dtype=torch.float16):
            values = []
            for chunk in images[mask].split(4):
                chunk = chunk.cuda()
                values.append(recompute(encoder, chunk, use_reentrant=False) if training else encoder(chunk))
            features = torch.cat(values)
            mask = mask.cuda()
            tokens = features.new_zeros(*mask.shape, 4, config['feature_dim'])
            tokens[mask] = features
            return head(tokens[None], mask[None], coords[None].cuda())
    encoder.train()
    head.train()
    total = 0.0
    steps = 0
    for start in range(0, len(train_index), 8):
        batch = train_index[start:start + 8]
        optimizer.zero_grad(set_to_none=True)
        for index in batch:
            loss = weighted_bce(predict(index, True), y[index:index + 1], w[index:index + 1])
            if not torch.isfinite(loss):
                raise ValueError('Non-finite adaptation loss.')
            scaler.scale(loss / len(batch)).backward()
            total += loss.item()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([*encoder.parameters(), *head.parameters()], 5.0)
        scaler.step(optimizer)
        scaler.update()
        steps += 1
        if start % 80 == 0:
            print(f'full FT epoch={epoch} segment={segment + 1}/4 studies={start}/{len(train_index)}', flush=True)
    validation = None
    if segment == 3:
        encoder.eval()
        head.eval()
        numerator = denominator = 0.0
        with torch.inference_mode():
            for index in valid:
                mass = w[index:index + 1].sum().item()
                numerator += weighted_bce(predict(index, False), y[index:index + 1], w[index:index + 1]).item() * mass
                denominator += mass
        validation = numerator / denominator
        if not np.isfinite(validation):
            raise ValueError('Non-finite weak validation loss.')
        if validation < best:
            best, best_encoder, best_epoch = (validation, cpu_tree(encoder.state_dict()), epoch)
    history.append(dict(epoch=epoch, segment=segment, studies=len(train_index), optimizer_steps=steps, train_loss=total / len(train_index), weak_validation_bce=validation, seconds=time.monotonic() - started))
    record = dict(recipe=adapt_recipe(config), origin_sha256=origin, epoch=epoch, segment=segment, encoder=cpu_tree(encoder.state_dict()), head=cpu_tree(head.state_dict()), optimizer=cpu_tree(optimizer.state_dict()), scaler=scaler.state_dict(), rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all(), best_score=best, best_encoder=best_encoder, best_epoch=best_epoch, history=history)
    torch.save(record, out / 'checkpoint.pt')
    progress = dict(recipe=record['recipe'], epoch=epoch, segment=segment, complete=True, checkpoint_sha256=file_hash(out / 'checkpoint.pt'), origin_sha256=origin, gold_studies_used_for_backbone=0, training_weak_studies=len(weak_finetune_split(gold, groups)[0]), validation_weak_studies=len(valid), trainable_parameters=sum((p.numel() for p in encoder.parameters())), best_epoch=best_epoch, validation_bce=None if best == float('inf') else best, history=history)
    (out / 'progress.json').write_text(json.dumps(progress, indent=2), encoding='utf-8')


In [ ]:
"""Exact saved preprocessing/head; no online weights or rank fitting on test data."""
import subprocess
import sys
import time

def load_anatomy_predictor(bundle, config, device):
    meta = validate_weak_bundle(bundle, config)
    if meta['python'] != list(sys.version_info[:2]) or meta['torch'] != torch.__version__:
        raise ValueError('Use the pinned training runtime.')
    if config.get('ensemble'):
        members = []
        for eid in config['train_dependencies']:
            path = bundle / 'members' / eid
            cfg = json.loads((path / 'assets.json').read_text())['experiment']
            members.append(load_anatomy_predictor(path, cfg, device))
        return dict(config=config, members=members)
    return dict(config=config, encoder=torch.jit.load(str(bundle / 'encoder.ts'), map_location=device).eval(), head=load_anatomy_head(bundle / 'model.pt', config, device), reference=torch.load(bundle / 'reference.pt', map_location='cpu', weights_only=True).numpy())

@torch.inference_mode()
def anatomy_probability(predictor, uid, series, image_root, device, flip=False):
    cfg = predictor['config']
    if cfg.get('ensemble'):
        if cfg['method'] == 'tta':
            member = predictor['members'][0]
            return (anatomy_probability(member, uid, series, image_root, device, False) + anatomy_probability(member, uid, series, image_root, device, True)) / 2
        values = []
        for member in predictor['members']:
            p = anatomy_probability(member, uid, series, image_root, device)
            if cfg['method'] == 'rank':
                p = reference_cdf(member['reference'], p[None])[0]
            values.append(p)
        return np.mean(values, axis=0)
    tokens, mask, coords = anatomy_tokens(uid, series, image_root, predictor['encoder'], cfg, device, flip=flip)
    return predictor['head'](tokens[None].to(device), mask[None].to(device), coords[None].to(device)).sigmoid().cpu().numpy()[0]

def submit_anatomy_experiment(config, data, bundle, out):
    data, bundle, out = (Path(data), Path(bundle), Path(out))
    started = time.monotonic()
    validate_weak_bundle(bundle, config)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps', '--find-links', str(bundle / 'wheels'), 'pydicom==3.0.2', 'python-gdcm==3.2.6'])
    torch.set_num_threads(1)
    torch.backends.mha.set_fastpath_enabled(False)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    predictor = load_anatomy_predictor(bundle, config, device)
    test = pd.read_csv(data / 'test.csv')
    series = pd.read_csv(data / 'test_series.csv')
    columns = pd.read_csv(data / 'sample_submission.csv', nrows=0).columns
    if not len(test) or test.StudyInstanceUID.isna().any() or (not test.StudyInstanceUID.is_unique) or (not series.SeriesInstanceUID.is_unique):
        raise ValueError('Invalid test metadata.')
    probabilities = [anatomy_probability(predictor, uid, series, data / 'test_series', device) for uid in test.StudyInstanceUID]
    out.mkdir(parents=True, exist_ok=True)
    result = save_submission(test.StudyInstanceUID.tolist(), probabilities, columns, out / 'submission.csv')
    (out / 'inference_summary.json').write_text(json.dumps(dict(experiment=config, studies=len(test), labels=12, seconds=time.monotonic() - started, manifest_sha256=file_hash(bundle / 'assets.json'), internet_required=False), indent=2))
    return result


In [ ]:
train_anatomy_experiment(EXPERIMENT,find_data(),Path('/kaggle/working/run'))
